In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT / "Processed Data" / "ev_charging_cleaned.csv"
)

MODEL_PATH = (
    PROJECT_ROOT / "models" / "lightgbm_demand_model.pkl"
)

file_path = DATA_PATH

df = pd.read_csv(file_path)
print(df.shape)
print(df.head())
print(df.info())

(72826, 18)
   UserID  ChargerID  ChargerCompany Location  ChargerType    StartDay  \
0       0          1               1    hotel            0  2022-09-15   
1       0          1               1    hotel            0  2022-09-14   
2       0          1               1    hotel            0  2022-09-14   
3       0          1               1    hotel            0  2022-09-29   
4       0          1               1    hotel            0  2022-09-25   

  StartTime      EndDay   EndTime        StartDatetime          EndDatetime  \
0  20:54:02  2022-09-15  23:59:13  2022-09-15 20:54:00  2022-09-15 23:59:00   
1  20:01:05  2022-09-14  21:31:04  2022-09-14 20:01:00  2022-09-14 21:31:00   
2  18:54:30  2022-09-14  19:54:29  2022-09-14 18:54:00  2022-09-14 19:54:00   
3  18:32:51  2022-09-30   0:16:42  2022-09-29 18:32:00  2022-09-30 00:16:00   
4  19:30:15  2022-09-26   0:30:14  2022-09-25 19:30:00  2022-09-26 00:30:00   

   Duration  Demand  Calculated_Duration  Duration_Difference Durati

In [16]:
#convert date time
df['StartDatetime'] = pd.to_datetime(df['StartDatetime'])
print(df['StartDatetime'].dtype)

datetime64[us]


In [17]:
df['Month'] = df['StartDatetime'].dt.month
df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)
print(df[['StartDatetime', 'Hour', 'DayOfWeek', 'Month', 'IsWeekend']].head(10))

df['Quarter'] = df['StartDatetime'].dt.quarter
# Extract day of the month
df['Day'] = df['StartDatetime'].dt.day

# Create time-of-day feature

def get_time_of_day(hour):
    if 0 <= hour < 6:
        return 'Night'
    elif 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df['TimeOfDay'] = df['Hour'].apply(get_time_of_day)
print(df[['StartDatetime', 'Hour', 'Day', 'TimeOfDay']].head())

        StartDatetime  Hour  DayOfWeek  Month  IsWeekend
0 2022-09-15 20:54:00    20          3      9          0
1 2022-09-14 20:01:00    20          2      9          0
2 2022-09-14 18:54:00    18          2      9          0
3 2022-09-29 18:32:00    18          3      9          0
4 2022-09-25 19:30:00    19          6      9          1
5 2022-09-24 14:51:00    14          5      9          1
6 2022-08-11 18:28:00    18          3      8          0
7 2022-08-10 19:37:00    19          2      8          0
8 2022-08-09 19:43:00    19          1      8          0
9 2022-08-07 19:51:00    19          6      8          1
        StartDatetime  Hour  Day TimeOfDay
0 2022-09-15 20:54:00    20   15   Evening
1 2022-09-14 20:01:00    20   14   Evening
2 2022-09-14 18:54:00    18   14   Evening
3 2022-09-29 18:32:00    18   29   Evening
4 2022-09-25 19:30:00    19   25   Evening


In [18]:
print(df['Location'].unique())
print(df['ChargerType'].unique())
print(df['ChargerCompany'].unique())

<ArrowStringArray>
[             'hotel',            'company',          'apartment',
             'resort',        'public area', 'public institution',
         'restaurant',      'accommodation',               'golf',
        'sightseeing',             'market',            'camping',
 'public parking lot',         'bus garage']
Length: 14, dtype: str
[0 1]
[1 0]


In [19]:
# create features

from sklearn.model_selection import train_test_split

candidate_features = [
    'Hour',
    'Day',
    'DayOfWeek',
    'Month',
    'Quarter',
    'IsWeekend',
    'TimeOfDay',
    'Location',
    'ChargerType',
    'ChargerCompany',
    'Calculated_Duration'
]

X = df[candidate_features]
y = df['Demand']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (58260, 11)
X_test shape: (14566, 11)
y_train shape: (58260,)
y_test shape: (14566,)


In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = ['Location','TimeOfDay']

numeric_features = [
    'Hour',
    'Day',
    'DayOfWeek',
    'Month',
    'Quarter',
    'IsWeekend',
    'ChargerType',
    'ChargerCompany',
    'Calculated_Duration'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('numeric', 'passthrough', numeric_features)
    ]
)

#fit the model

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

#check the shapes
print("X_train transformed shape:", X_train_transformed.shape)
print("X_test transformed shape:", X_test_transformed.shape)

X_train transformed shape: (58260, 27)
X_test transformed shape: (14566, 27)


In [21]:
feature_names = preprocessor.get_feature_names_out()

print(feature_names)
print("Total features:", len(feature_names))

['categorical__Location_accommodation' 'categorical__Location_apartment'
 'categorical__Location_bus garage' 'categorical__Location_camping'
 'categorical__Location_company' 'categorical__Location_golf'
 'categorical__Location_hotel' 'categorical__Location_market'
 'categorical__Location_public area'
 'categorical__Location_public institution'
 'categorical__Location_public parking lot' 'categorical__Location_resort'
 'categorical__Location_restaurant' 'categorical__Location_sightseeing'
 'categorical__TimeOfDay_Afternoon' 'categorical__TimeOfDay_Evening'
 'categorical__TimeOfDay_Morning' 'categorical__TimeOfDay_Night'
 'numeric__Hour' 'numeric__Day' 'numeric__DayOfWeek' 'numeric__Month'
 'numeric__Quarter' 'numeric__IsWeekend' 'numeric__ChargerType'
 'numeric__ChargerCompany' 'numeric__Calculated_Duration']
Total features: 27


In [22]:
# for 2 models

# Model A: Pre-session prediction
features_model_a = [
    'Hour',
    'Day',
    'DayOfWeek',
    'Month',
    'Quarter',
    'IsWeekend',
    'TimeOfDay',
    'Location',
    'ChargerType',
    'ChargerCompany'
]

# Model B: Session-aware prediction
features_model_b = features_model_a + ['Calculated_Duration']

X_A = df[features_model_a]
X_B = df[features_model_b]

y = df['Demand']

print("Model A shape:", X_A.shape)
print("Model B shape:", X_B.shape)
print("Target shape:", y.shape)

print("Model A features:")
print(X_A.columns.tolist())

print("\nModel B features:")
print(X_B.columns.tolist())

Model A shape: (72826, 10)
Model B shape: (72826, 11)
Target shape: (72826,)
Model A features:
['Hour', 'Day', 'DayOfWeek', 'Month', 'Quarter', 'IsWeekend', 'TimeOfDay', 'Location', 'ChargerType', 'ChargerCompany']

Model B features:
['Hour', 'Day', 'DayOfWeek', 'Month', 'Quarter', 'IsWeekend', 'TimeOfDay', 'Location', 'ChargerType', 'ChargerCompany', 'Calculated_Duration']


In [23]:
# train the model
from sklearn.model_selection import train_test_split

# Model A split
X_A_train, X_A_test, y_A_train, y_A_test = train_test_split(
    X_A,
    y,
    test_size=0.2,
    random_state=42
)

# Model B split
X_B_train, X_B_test, y_B_train, y_B_test = train_test_split(
    X_B,
    y,
    test_size=0.2,
    random_state=42
)

print("Model A:")
print("X_train:", X_A_train.shape)
print("X_test:", X_A_test.shape)
print("y_train:", y_A_train.shape)
print("y_test:", y_A_test.shape)

print("\nModel B:")
print("X_train:", X_B_train.shape)
print("X_test:", X_B_test.shape)
print("y_train:", y_B_train.shape)
print("y_test:", y_B_test.shape)

Model A:
X_train: (58260, 10)
X_test: (14566, 10)
y_train: (58260,)
y_test: (14566,)

Model B:
X_train: (58260, 11)
X_test: (14566, 11)
y_train: (58260,)
y_test: (14566,)


In [24]:
# Categorical features
categorical_features = [
    'Location',
    'TimeOfDay'
]

# Numeric features for Model A
numeric_features_a = [
    'Hour',
    'Day',
    'DayOfWeek',
    'Month',
    'Quarter',
    'IsWeekend',
    'ChargerType',
    'ChargerCompany'
]

# Numeric features for Model B
numeric_features_b = numeric_features_a + ['Calculated_Duration']

#Create the two preprocessors

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor_a = ColumnTransformer(transformers=[('categorical',OneHotEncoder(handle_unknown='ignore'),categorical_features),
        ('numeric','passthrough',numeric_features_a)])

preprocessor_b = ColumnTransformer(
    transformers=[
        (
            'categorical',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_features
        ),
        (
            'numeric',
            'passthrough',
            numeric_features_b
        )
    ]
)

In [25]:
#Build Linear regression
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# Model A pipeline
linear_model_a = Pipeline([
    ('preprocessor', preprocessor_a),
    ('model', LinearRegression())
])

# Model B pipeline
linear_model_b = Pipeline([
    ('preprocessor', preprocessor_b),
    ('model', LinearRegression())
])

#fit both the models
# Train Model A
linear_model_a.fit(X_A_train, y_A_train)

# Train Model B
linear_model_b.fit(X_B_train, y_B_train)

print("Both Linear Regression models trained successfully.")

Both Linear Regression models trained successfully.


In [26]:
#predict the models
# Predictions for Model A
y_A_pred = linear_model_a.predict(X_A_test)

# Predictions for Model B
y_B_pred = linear_model_b.predict(X_B_test)

#calculate the metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Model A metrics
mae_a = mean_absolute_error(y_A_test, y_A_pred)
mse_a = mean_squared_error(y_A_test, y_A_pred)
rmse_a = np.sqrt(mse_a)
r2_a = r2_score(y_A_test, y_A_pred)

# Model B metrics
mae_b = mean_absolute_error(y_B_test, y_B_pred)
mse_b = mean_squared_error(y_B_test, y_B_pred)
rmse_b = np.sqrt(mse_b)
r2_b = r2_score(y_B_test, y_B_pred)

print("Linear Regression - Model A")
print("MAE :", mae_a)
print("MSE :", mse_a)
print("RMSE:", rmse_a)
print("R²  :", r2_a)

print("\nLinear Regression - Model B")
print("MAE :", mae_b)
print("MSE :", mse_b)
print("RMSE:", rmse_b)
print("R²  :", r2_b)

Linear Regression - Model A
MAE : 9.735913286918926
MSE : 164.07181064564716
RMSE: 12.809051902683787
R²  : 0.08949192398579664

Linear Regression - Model B
MAE : 6.260499344600485
MSE : 100.87504832192212
RMSE: 10.043657118894597
R²  : 0.44019910669603


In [27]:
# Decision Tree
from sklearn.tree import DecisionTreeRegressor

tree_model_a = Pipeline([
    ('preprocessor', preprocessor_a),
    ('model', DecisionTreeRegressor(random_state=42))
])

tree_model_b = Pipeline([
    ('preprocessor', preprocessor_b),
    ('model', DecisionTreeRegressor(random_state=42))
])

#fit the model
tree_model_a.fit(X_A_train, y_A_train)
tree_model_b.fit(X_B_train, y_B_train)

print("Decision Tree models trained successfully.")

Decision Tree models trained successfully.


In [28]:
#Evaluate the models
# Predictions
y_tree_A_pred = tree_model_a.predict(X_A_test)
y_tree_B_pred = tree_model_b.predict(X_B_test)

# Model A metrics
tree_mae_a = mean_absolute_error(y_A_test, y_tree_A_pred)
tree_mse_a = mean_squared_error(y_A_test, y_tree_A_pred)
tree_rmse_a = np.sqrt(tree_mse_a)
tree_r2_a = r2_score(y_A_test, y_tree_A_pred)

# Model B metrics
tree_mae_b = mean_absolute_error(y_B_test, y_tree_B_pred)
tree_mse_b = mean_squared_error(y_B_test, y_tree_B_pred)
tree_rmse_b = np.sqrt(tree_mse_b)
tree_r2_b = r2_score(y_B_test, y_tree_B_pred)

print("Decision Tree - Model A")
print("MAE:", tree_mae_a)
print("MSE:", tree_mse_a)
print("RMSE:", tree_rmse_a)
print("R²:", tree_r2_a)

print("\nDecision Tree - Model B")
print("MAE:", tree_mae_b)
print("MSE:", tree_mse_b)
print("RMSE:", tree_rmse_b)
print("R²:", tree_r2_b)

Decision Tree - Model A
MAE: 12.4815115884676
MSE: 286.4100704683035
RMSE: 16.923654170075196
R²: -0.5894179578258096

Decision Tree - Model B
MAE: 4.452057577005813
MSE: 73.95344281991548
RMSE: 8.599618760149514
R²: 0.5895991720234359


In [29]:
# random Forest
from sklearn.ensemble import RandomForestRegressor

rf_model_a = Pipeline([
    ('preprocessor', preprocessor_a),
    ('model', RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model_b = Pipeline([
    ('preprocessor', preprocessor_b),
    ('model', RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

# Fit the model
rf_model_a.fit(X_A_train, y_A_train)
rf_model_b.fit(X_B_train, y_B_train)

print("Random Forest models trained successfully.")

Random Forest models trained successfully.


In [30]:
# Evaluate
# Predictions
y_rf_A_pred = rf_model_a.predict(X_A_test)
y_rf_B_pred = rf_model_b.predict(X_B_test)

# Model A metrics
rf_mae_a = mean_absolute_error(y_A_test, y_rf_A_pred)
rf_mse_a = mean_squared_error(y_A_test, y_rf_A_pred)
rf_rmse_a = np.sqrt(rf_mse_a)
rf_r2_a = r2_score(y_A_test, y_rf_A_pred)

# Model B metrics
rf_mae_b = mean_absolute_error(y_B_test, y_rf_B_pred)
rf_mse_b = mean_squared_error(y_B_test, y_rf_B_pred)
rf_rmse_b = np.sqrt(rf_mse_b)
rf_r2_b = r2_score(y_B_test, y_rf_B_pred)

print("Random Forest - Model A")
print("MAE:", rf_mae_a)
print("MSE:", rf_mse_a)
print("RMSE:", rf_rmse_a)
print("R²:", rf_r2_a)

print("\nRandom Forest - Model B")
print("MAE:", rf_mae_b)
print("MSE:", rf_mse_b)
print("RMSE:", rf_rmse_b)
print("R²:", rf_r2_b)


Random Forest - Model A
MAE: 10.392298055004007
MSE: 191.05399504456196
RMSE: 13.822228295197629
R²: -0.06024432081481579

Random Forest - Model B
MAE: 3.508224064035001
MSE: 38.86655966812822
RMSE: 6.234305066976448
R²: 0.784312025780295


In [31]:
#Gradient Boosting
from sklearn.ensemble import GradientBoostingRegressor

gb_model_a = Pipeline([
    ('preprocessor', preprocessor_a),
    ('model', GradientBoostingRegressor(
        n_estimators=100,
        random_state=42
    ))
])

gb_model_b = Pipeline([
    ('preprocessor', preprocessor_b),
    ('model', GradientBoostingRegressor(
        n_estimators=100,
        random_state=42
    ))
])

#fit the model
gb_model_a.fit(X_A_train, y_A_train)
gb_model_b.fit(X_B_train, y_B_train)

print("Gradient Boosting models trained successfully.")

Gradient Boosting models trained successfully.


In [32]:
# Evaluate

# Predictions
y_gb_A_pred = gb_model_a.predict(X_A_test)
y_gb_B_pred = gb_model_b.predict(X_B_test)

# Model A metrics
gb_mae_a = mean_absolute_error(y_A_test, y_gb_A_pred)
gb_mse_a = mean_squared_error(y_A_test, y_gb_A_pred)
gb_rmse_a = np.sqrt(gb_mse_a)
gb_r2_a = r2_score(y_A_test, y_gb_A_pred)

# Model B metrics
gb_mae_b = mean_absolute_error(y_B_test, y_gb_B_pred)
gb_mse_b = mean_squared_error(y_B_test, y_gb_B_pred)
gb_rmse_b = np.sqrt(gb_mse_b)
gb_r2_b = r2_score(y_B_test, y_gb_B_pred)

print("Gradient Boosting - Model A")
print("MAE:", gb_mae_a)
print("MSE:", gb_mse_a)
print("RMSE:", gb_rmse_a)
print("R²:", gb_r2_a)

print("\nGradient Boosting - Model B")
print("MAE:", gb_mae_b)
print("MSE:", gb_mse_b)
print("RMSE:", gb_rmse_b)
print("R²:", gb_r2_b)

Gradient Boosting - Model A
MAE: 9.452291461198495
MSE: 156.76423668669418
RMSE: 12.520552571140547
R²: 0.1300449298892169

Gradient Boosting - Model B
MAE: 3.7775080739317892
MSE: 42.09961490377028
RMSE: 6.4884216034233075
R²: 0.7663703520054514


In [34]:
from xgboost import XGBRegressor

xgb_model_a = Pipeline([
    ('preprocessor', preprocessor_a),
    ('model', XGBRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

xgb_model_b = Pipeline([
    ('preprocessor', preprocessor_b),
    ('model', XGBRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

#fit the model
xgb_model_a.fit(X_A_train, y_A_train)
xgb_model_b.fit(X_B_train, y_B_train)

print("XGBoost models trained successfully.")

XGBoost models trained successfully.


In [37]:
#Evaluation
# Predictions
y_xgb_A_pred = xgb_model_a.predict(X_A_test)
y_xgb_B_pred = xgb_model_b.predict(X_B_test)

# Model A metrics
xgb_mae_a = mean_absolute_error(y_A_test, y_xgb_A_pred)
xgb_mse_a = mean_squared_error(y_A_test, y_xgb_A_pred)
xgb_rmse_a = np.sqrt(xgb_mse_a)
xgb_r2_a = r2_score(y_A_test, y_xgb_A_pred)

# Model B metrics
xgb_mae_b = mean_absolute_error(y_B_test, y_xgb_B_pred)
xgb_mse_b = mean_squared_error(y_B_test, y_xgb_B_pred)
xgb_rmse_b = np.sqrt(xgb_mse_b)
xgb_r2_b = r2_score(y_B_test, y_xgb_B_pred)

print("XGBoost - Model A")
print("MAE:", xgb_mae_a)
print("MSE:", xgb_mse_a)
print("RMSE:", xgb_rmse_a)
print("R²:", xgb_r2_a)

print("XGBoost - Model B")
print("MAE:", xgb_mae_b)
print("MSE:", xgb_mse_b)
print("RMSE:", xgb_rmse_b)
print("R²:", xgb_r2_b)

XGBoost - Model A
MAE: 9.411917335493754
MSE: 156.15307020112422
RMSE: 12.496122206553688
R²: 0.13343656687250405
XGBoost - Model B
MAE: 3.4533297492160364
MSE: 37.337201954686186
RMSE: 6.110417494303166
R²: 0.7927991177659562


In [39]:
from lightgbm import LGBMRegressor

print("LightGBM imported successfully.")

LightGBM imported successfully.


In [40]:
#Lightgbm model

from lightgbm import LGBMRegressor

lgbm_model_a = Pipeline([
    ('preprocessor', preprocessor_a),
    ('model', LGBMRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
])

lgbm_model_b = Pipeline([
    ('preprocessor', preprocessor_b),
    ('model', LGBMRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
])

#fit the model
lgbm_model_a.fit(X_A_train, y_A_train)
lgbm_model_b.fit(X_B_train, y_B_train)

print("LightGBM models trained successfully.")

LightGBM models trained successfully.


In [41]:
#Evaluate
# Predictions
y_lgbm_A_pred = lgbm_model_a.predict(X_A_test)
y_lgbm_B_pred = lgbm_model_b.predict(X_B_test)

# Model A metrics
lgbm_mae_a = mean_absolute_error(y_A_test, y_lgbm_A_pred)
lgbm_mse_a = mean_squared_error(y_A_test, y_lgbm_A_pred)
lgbm_rmse_a = np.sqrt(lgbm_mse_a)
lgbm_r2_a = r2_score(y_A_test, y_lgbm_A_pred)

# Model B metrics
lgbm_mae_b = mean_absolute_error(y_B_test, y_lgbm_B_pred)
lgbm_mse_b = mean_squared_error(y_B_test, y_lgbm_B_pred)
lgbm_rmse_b = np.sqrt(lgbm_mse_b)
lgbm_r2_b = r2_score(y_B_test, y_lgbm_B_pred)

print("LightGBM - Model A")
print("MAE:", lgbm_mae_a)
print("MSE:", lgbm_mse_a)
print("RMSE:", lgbm_rmse_a)
print("R²:", lgbm_r2_a)

print("\nLightGBM - Model B")
print("MAE:", lgbm_mae_b)
print("MSE:", lgbm_mse_b)
print("RMSE:", lgbm_rmse_b)
print("R²:", lgbm_r2_b)

LightGBM - Model A
MAE: 9.33500349632449
MSE: 153.48501913018612
RMSE: 12.388907099909423
R²: 0.14824277908987338

LightGBM - Model B
MAE: 3.4909768572274906
MSE: 37.20983053109268
RMSE: 6.099986109090141
R²: 0.7935059589312907


In [42]:
#Comparing the results

comparison_results = pd.DataFrame({
    'Model': [
        'Linear Regression', 'Linear Regression',
        'Decision Tree', 'Decision Tree',
        'Random Forest', 'Random Forest',
        'Gradient Boosting', 'Gradient Boosting',
        'XGBoost', 'XGBoost',
        'LightGBM', 'LightGBM'
    ],
    'Scenario': [
        'Model A', 'Model B',
        'Model A', 'Model B',
        'Model A', 'Model B',
        'Model A', 'Model B',
        'Model A', 'Model B',
        'Model A', 'Model B'
    ],
    'MAE': [
        mae_a, mae_b,
        tree_mae_a, tree_mae_b,
        rf_mae_a, rf_mae_b,
        gb_mae_a, gb_mae_b,
        xgb_mae_a, xgb_mae_b,
        lgbm_mae_a, lgbm_mae_b
    ],
    'MSE': [
        mse_a, mse_b,
        tree_mse_a, tree_mse_b,
        rf_mse_a, rf_mse_b,
        gb_mse_a, gb_mse_b,
        xgb_mse_a, xgb_mse_b,
        lgbm_mse_a, lgbm_mse_b
    ],
    'RMSE': [
        rmse_a, rmse_b,
        tree_rmse_a, tree_rmse_b,
        rf_rmse_a, rf_rmse_b,
        gb_rmse_a, gb_rmse_b,
        xgb_rmse_a, xgb_rmse_b,
        lgbm_rmse_a, lgbm_rmse_b
    ],
    'R2': [
        r2_a, r2_b,
        tree_r2_a, tree_r2_b,
        rf_r2_a, rf_r2_b,
        gb_r2_a, gb_r2_b,
        xgb_r2_a, xgb_r2_b,
        lgbm_r2_a, lgbm_r2_b
    ]
})

comparison_results

,Model,Scenario,MAE,MSE,RMSE,R2
0,Linear Regression,Model A,9.735913,164.071811,12.809052,0.089492
1,Linear Regression,Model B,6.260499,100.875048,10.043657,0.440199
2,Decision Tree,Model A,12.481512,286.410070,16.923654,-0.589418
3,Decision Tree,Model B,4.452058,73.953443,8.599619,0.589599
4,Random Forest,Model A,10.392298,191.053995,13.822228,-0.060244
5,Random Forest,Model B,3.508224,38.866560,6.234305,0.784312
6,Gradient Boosting,Model A,9.452291,156.764237,12.520553,0.130045
7,Gradient Boosting,Model B,3.777508,42.099615,6.488422,0.766370
8,XGBoost,Model A,9.411917,156.153070,12.496122,0.133437
9,XGBoost,Model B,3.453330,37.337202,6.110417,0.792799


In [43]:
# Identify the important factors influencing charging demand.

feature_names = lgbm_model_b.named_steps['preprocessor'].get_feature_names_out()

feature_importance = lgbm_model_b.named_steps['model'].feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
})

importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
)

importance_df.head(15)

,Feature,Importance
26,numeric__Calculated_Duration,1095
21,numeric__Month,378
18,numeric__Hour,361
19,numeric__Day,225
25,numeric__ChargerCompany,154
24,numeric__ChargerType,115
20,numeric__DayOfWeek,106
1,categorical__Location_apartment,93
6,categorical__Location_hotel,58
7,categorical__Location_market,57


Calculated session duration was the most influential feature in predicting charging demand, followed by month, hour, day, charger company, and charger type.

In [44]:
#validation before pickling
# Create a validation DataFrame
validation_df = pd.DataFrame({
    'Actual_Demand': y_B_test.values,
    'Predicted_Demand': y_lgbm_B_pred
})

# Calculate prediction error
validation_df['Error'] = (
    validation_df['Actual_Demand'] -
    validation_df['Predicted_Demand']
)

validation_df.head(10)

,Actual_Demand,Predicted_Demand,Error
0,10.32,13.413813,-3.093813
1,30.40,26.532688,3.867312
2,15.00,13.197051,1.802949
3,13.18,11.734434,1.445566
4,42.30,39.499727,2.800273
5,0.11,0.475612,-0.365612
6,17.20,14.853750,2.346250
7,30.31,28.577681,1.732319
8,22.53,21.225268,1.304732
9,11.00,29.700729,-18.700729


In [ ]:
#pickling

import pickle

model_path = MODEL_PATH

with open(model_path, 'wb') as file:
    pickle.dump(lgbm_model_b, file)

print("Model saved successfully.")

Model saved successfully.


In [ ]:
import pickle

model_path = MODEL_PATH

with open(model_path, 'rb') as file:
    loaded_model = pickle.load(file)

print("Model loaded successfully.")

Model loaded successfully.


In [47]:
# final verification

# Use the first test record as a sample input
sample_input = X_B_test.iloc[[0]]

# Predict using the loaded pickle model
sample_prediction = loaded_model.predict(sample_input)

print("Predicted Demand:", sample_prediction[0])
print("Actual Demand:", y_B_test.iloc[0])

Predicted Demand: 13.413813318643486
Actual Demand: 10.32
